# 00 – Colab Bootstrap (einmal pro Session)

**Code kommt von GitHub** (`git clone` / `git pull`) — **nur CSVs liegen auf Drive.**

1. Einmalig auf Drive: CSVs in `.../DataMining/data/` **oder** `.../DataMining_Final-Project/data/`
2. Jede Colab-Session: diese Zelle ausführen
3. Danach: `03_preprocessing` → `04_modeling` (aus dem geklonten Repo)

Nach `git push` am Mac: hier **Zelle neu ausführen** → `git pull` holt Updates.

In [ ]:
# Erste Ausführung: clone von GitHub
# Später: nur git pull

import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/jspldrch/DataMining_Final-Project.git"
BRANCH = "main"
REPO_DIR = Path("/content/DataMining_Final-Project")
DRIVE_OUTPUTS = Path("/content/drive/MyDrive/DataMining/outputs")

DRIVE_DATA_CANDIDATES = [
    Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project/data"),
    Path("/content/drive/MyDrive/DataMining/data"),
]
DRIVE_DATA = None
for p in DRIVE_DATA_CANDIDATES:
    if (p / "train.csv").exists() and (p / "test.csv").exists():
        DRIVE_DATA = p
        break

drive.mount("/content/drive")

if (REPO_DIR / ".git").exists():
    !cd "{REPO_DIR}" && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

!pip install -q -r requirements.txt

if DRIVE_DATA is None:
    tried = "\n".join(f"  - {p}" for p in DRIVE_DATA_CANDIDATES)
    raise FileNotFoundError(f"train.csv/test.csv nicht auf Drive.\nGeprüft:\n{tried}")

# Outputs auf Drive (Parquet + Submission überleben Session-Ende)
for name in ("processed", "submissions", "figures", "regional"):
    (DRIVE_OUTPUTS / name).mkdir(parents=True, exist_ok=True)
    link = REPO_DIR / "outputs" / name
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    if not link.exists():
        link.symlink(DRIVE_OUTPUTS / name, target_is_directory=True)

TRAIN_PATH = DRIVE_DATA / "train.csv"
TEST_PATH = DRIVE_DATA / "test.csv"
PROCESSED_DIR = REPO_DIR / "outputs" / "processed"

print("✓ Bootstrap OK")
print("  Code:", REPO_DIR)
print("  Daten:", DRIVE_DATA)
print("  Outputs:", PROCESSED_DIR)
print("\n→ Datei öffnen: /content/DataMining_Final-Project/notebooks/03_preprocessing.ipynb")